# Stock Price Movement Analysis

Educational / research notebook for **next-day direction classification** on historical CBX (Zagreb Stock Exchange) prices.

**Not financial advice.** This project does not forecast price levels, execute trades, or recommend investments.

The original exploratory notebook is preserved at `notebooks/original_stock_price_prediction.ipynb`.


## 1. Setup

Uses the project `src/` package for reproducible data loading, feature engineering, training, and evaluation.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import dataset_metadata, ensure_local_csv, load_and_prepare, load_raw_data
from src.evaluate import generate_all_figures, evaluate_models, save_metrics
from src.features import FEATURE_COLUMNS, TARGET_COLUMN, build_feature_frame, get_feature_matrix
from src.train import chronological_train_test_split, run_training_pipeline, train_models

pd.set_option("display.max_columns", 20)
print("Project root:", PROJECT_ROOT)


## 2. Data source

- **Source:** Kaggle dataset [`satyajeetbedi/stock-price-dataset`](https://www.kaggle.com/datasets/satyajeetbedi/stock-price-dataset)
- **Series:** symbol `CBX`, MIC `XZAG` (Zagreb Stock Exchange), ISIN `HRZB00ICBEX6`
- **Fields used:** `date`, `open_value`, `high_value`, `low_value`, `last_value`


In [ ]:
csv_path = ensure_local_csv()
raw = load_raw_data(csv_path)
prices = load_and_prepare(csv_path)
meta = dataset_metadata(prices, raw)
meta


In [ ]:
prices.head()


## 3. Target definition

$$
\text{Target}_t = \mathbb{1}\![\text{last\_value}_{t+1} > \text{last\_value}_t]
$$

This is **direction classification**, not price-level forecasting.

**Leakage control:** the final row has no next close and is left as missing, then dropped (the original notebook coerced that comparison to class `0`).


## 4. Feature engineering

Preserved from the original notebook:

| Feature | Definition |
|---------|------------|
| `open_value`, `high_value`, `low_value`, `last_value` | Same-day OHLC |
| `MA10`, `MA50` | Trailing moving averages of `last_value` |
| `Return` | Daily percent change of `last_value` |

Rolling windows use only historical observations. Incomplete warm-up rows are dropped.


In [ ]:
featured = build_feature_frame(prices)
print("Featured shape:", featured.shape)
print("Date range:", featured["date"].min().date(), "→", featured["date"].max().date())
print("Class balance:")
print(featured[TARGET_COLUMN].value_counts(normalize=True).rename("proportion"))
featured.head()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(featured["date"], featured["last_value"], color="#1f4e79", lw=1.2)
ax.set_title("CBX Closing Price History")
ax.set_xlabel("Date")
ax.set_ylabel("Last value")
ax.grid(True, alpha=0.3)
plt.show()


## 5. Chronological train/test split

**Leakage control:** `shuffle=False` / explicit time ordering. Training dates always precede test dates. No random split.


In [ ]:
X_train, X_test, y_train, y_test, train_dates, test_dates = chronological_train_test_split(
    featured, test_size=0.2
)
print(f"Train: {len(X_train)} rows ({train_dates.min().date()} → {train_dates.max().date()})")
print(f"Test:  {len(X_test)} rows ({test_dates.min().date()} → {test_dates.max().date()})")
assert train_dates.max() < test_dates.min()
assert list(X_train.columns) == FEATURE_COLUMNS
assert TARGET_COLUMN not in X_train.columns


## 6. Models

Primary model (original): **Random Forest** (`n_estimators=200`).

Baselines for comparison only:

- Majority-class baseline (predict most frequent training label)
- Logistic regression (with train-only `StandardScaler` inside a Pipeline)


In [ ]:
artifacts = train_models(X_train, y_train)
results = evaluate_models(artifacts, X_test, y_test, y_train)

for name in ("majority_class", "logistic_regression", "random_forest"):
    m = results[name]
    roc = m.get("roc_auc")
    roc_str = f"{roc:.4f}" if roc is not None else "n/a"
    print(
        f"{name:22s}  acc={m['accuracy']:.4f}  "
        f"precision={m['precision']:.4f}  recall={m['recall']:.4f}  "
        f"f1={m['f1']:.4f}  roc_auc={roc_str}"
    )


In [ ]:
import json
print(json.dumps(results["random_forest"]["classification_report"], indent=2))


## 7. Feature importance (Random Forest)


In [ ]:
fi = pd.DataFrame(results["feature_importance"])
print(fi.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4))
fi_sorted = fi.sort_values("Importance")
ax.barh(fi_sorted["Feature"], fi_sorted["Importance"], color="#1f4e79")
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Importance")
ax.grid(True, axis="x", alpha=0.3)
plt.show()


## 8. Persist artifacts and figures

Writes models, metrics, and PNGs under `models/` and `docs/images/`.


In [ ]:
# Full reproducible pipeline (train + evaluate + figures)
train_result = run_training_pipeline(data_path=csv_path)
# Reuse freshly trained artifacts already evaluated above via package helpers
from src.evaluate import run_evaluation

evaluation = run_evaluation(train_result)
print("Metrics file:", evaluation["metrics_path"])
for p in evaluation["figure_paths"]:
    print("Figure:", p)


## 9. Interpretation and limitations

Test-set accuracy for the Random Forest is typically **near 50%** on this series — close to a coin flip / majority baseline. That does **not** support claims of reliable market prediction.

### Limitations

- **Market noise:** daily direction is weakly predictable from simple technical features.
- **Non-stationarity:** market regimes change; past relationships may not hold.
- **Transaction costs:** even slight edge can vanish after costs and slippage (not modeled).
- **Class imbalance:** mild imbalance can inflate naive baselines.
- **Overfitting:** tree ensembles can memorize training patterns that fail out of sample.
- **Data leakage risk:** mitigated here via chronological splits and trailing indicators, but always re-check when adding features.
- **No causal interpretation:** feature importance ≠ economic causation.
- **Past performance ≠ future performance.**

### Disclaimer

This project is for educational and research purposes only. It is not financial advice and should not be used to make investment decisions.
